## **Approach 3: Training from Scratch using Open Datasets (Custom Modeling)**

Reasons
* **Open Dataset**: Aapne internet par pehle se maujood ek public dataset (torchaudio.datasets.YESNO) use kiya, yaani aapko khud se audio records nahi karne pade.

* **Custom Feature Extraction**: Aapne raw audio lekar khud uska Mel-Spectrogram banaya aur mathematically uska mean lekar custom features (X) design kiye.

* **Training From Scratch**: Aapne LogisticRegression() model liya jise aawaz ke baare mein pehle se kuch nahi pata tha (zero pre-trained weights). Usne bilkul zero se (from scratch) aapke diye hue features aur labels (y) ke beech ke patterns ko seekha aur train hua.

## loading libraries & downloading dataset

In [ ]:
import os
import torch
import torchaudio

# 1. Pehle 'data' naam ka folder manually create karein agar wo nahi bana hua
os.makedirs("./data", exist_ok=True)

# 2. Ab dataset download karein
dataset = torchaudio.datasets.YESNO(root="./data", download=True)

print(f"Total Audio Files: {len(dataset)}")

100%|██████████| 4.49M/4.49M [00:00<00:00, 4.77MB/s]


Total Audio Files: 60


## Feature Extraction
Hum har audio file ko load karenge, uska numerical data (waveform) nikalenge, aur usay ek fix-size feature vector (mean values) mein badal denge taake ML model ko feed kar sakein.

In [ ]:
X = []
y = []

# Audio ko spectrogram (matrix) mein badalne ke liye torchaudio ka tool
mel_transform = torchaudio.transforms.MelSpectrogram(sample_rate=8000, n_mels=8)

for i in range(len(dataset)):
    waveform, sample_rate, labels = dataset[i]

    # 1. Audio ka Mel-Spectrogram (Numerical Matrix) banayein
    # Is se waveform badal kar [1, 8, time_steps] ka matrix ban jayegi
    spectrogram = mel_transform(waveform)

    # 2. Time steps ka mean le kar fixed size feature vector banayein (dim=2 ab valid hai kyunki data 3D ho gaya)
    features = torch.mean(spectrogram, dim=2).squeeze().numpy()

    X.append(features)

    # Pehla lafz kya bola (1 = Yes, 0 = No)
    y.append(labels[0])

X = np.array(X)
y = np.array(y)

print(f"Features Shape (X): {X.shape}") # Should be (60, 8) -> 60 files, 8 features each
print(f"Labels Shape (y): {y.shape}")

Features Shape (X): (60, 8)
Labels Shape (y): (60,)


## Train test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

Training samples: 48, Testing samples: 12


# Model Training & Accuracy

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Model initialize aur train karein
model = LogisticRegression()
model.fit(X_train, y_train)

# Predictions nikalyein
predictions = model.predict(X_test)

# Accuracy check karein
accuracy = accuracy_score(y_test, predictions)
print(f"🔥 Model Accuracy: {accuracy * 100}%")

🔥 Model Accuracy: 58.333333333333336%


## Testing

In [ ]:
# Test set se pehli file uthayein
sample_audio = X_test[0].reshape(1, -1)
true_label = "Yes" if y_test[0] == 1 else "No"

# Predict
predicted_label = "Yes" if model.predict(sample_audio)[0] == 1 else "No"

print(f"Asal mein word kya tha: {true_label}")
print(f"Model ne kya pehchana: {predicted_label}")

Asal mein word kya tha: No
Model ne kya pehchana: No
